<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/02_inverse_kinematics_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K2 2リンク平面アームの逆運動学(解析解)(notebook)

本文:[`02-inverse-kinematics.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/kinematics/02-inverse-kinematics.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`kinematics/02_inverse_kinematics.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

l1, l2 = 1.0, 0.8   # リンクの長さ(K1 と同じ)

def fk_points(q, l1=l1, l2=l2):
    # K1 の順運動学:肘の位置と手先の位置を返す
    q1, q2 = q
    elbow = l1 * np.array([np.cos(q1), np.sin(q1)])
    hand = elbow + l2 * np.array([np.cos(q1 + q2), np.sin(q1 + q2)])
    return elbow, hand

def draw_arm(ax, elbow, hand, **kw):
    # ベース(原点)・肘・手先を線で結んで、アームを描く
    pts = np.array([[0.0, 0.0], elbow, hand])
    ax.plot(pts[:, 0], pts[:, 1], "-o", **kw)

## 1. 予想と手計算

**予想**:目標 $(1.0, 0.8)$ に手先を置く関節角は、いくつあるか。アームの図を描いて予想する。

- 予想:(ここに書く)

**手計算**:目標 $(1.0, 0.8)$ について、本文の式 (1) の $c$ を計算し、2つの解を求める。1つは K1 で見た $q = (0, \pi/2)$ である(本文の確認課題1)。

- 手計算:(ここに書く)

## 2. $\cos q_2$ を計算する

**実装**:本文の式 (1) の右辺 $c$ を返す関数 `cos_q2` を書く。

In [ ]:
def cos_q2(p, l1=l1, l2=l2):
    x, y = p
    return (x**2 + y**2 - l1**2 - l2**2) / (2 * l1 * l2)

check("目標 (1.0, 0.8)", cos_q2([1.0, 0.8]), 0.0)
check("目標 (1.8, 0.0)(まっすぐ伸ばした姿勢)", cos_q2([1.8, 0.0]), 1.0)
check("目標 (2.5, 0.0)(届かない)", cos_q2([2.5, 0.0]), (6.25 - 1.64) / 1.6)

## 3. 逆運動学を解く

**実装**:関節角を返す関数 `ik` を書く。`sign=+1` なら $q_2 > 0$ の解、`sign=-1` なら $q_2 < 0$ の解を返す。$|c| > 1$ のときは `None` を返す。$q_1$ は本文の式 (2) で求める。

`check` は、関節角を答えと比べるだけでなく、**順運動学に戻して目標になるか**も確かめている(本文 §4)。

In [ ]:
def ik(p, sign=+1, l1=l1, l2=l2):
    # sign=+1 なら q2 > 0 の解、sign=-1 なら q2 < 0 の解を返す。届かないときは None
    x, y = p
    c = cos_q2(p, l1, l2)
    if abs(c) > 1:
        return None
    q2 = sign * np.arccos(c)
    q1 = np.arctan2(y, x) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2))
    return np.array([q1, q2])

check("目標 (1.0, 0.8)、q2 > 0 の解", ik([1.0, 0.8], +1), [0.0, np.pi / 2])
q_minus = ik([1.0, 0.8], -1)
check("目標 (1.0, 0.8)、q2 < 0 の解の q2", q_minus[1], -np.pi / 2)
check("q2 < 0 の解を順運動学に戻すと目標になる", fk_points(q_minus)[1], [1.0, 0.8])
check("(2.5, 0.0) は解なし(None)", ik([2.5, 0.0]) is None, True)
print("q2 < 0 の解:", np.round(q_minus, 4))

# ランダムな姿勢から手先の位置を作り、逆運動学で戻せるかを確かめる
rng = np.random.default_rng(0)
qs = np.column_stack([rng.uniform(-np.pi, np.pi, 200), rng.uniform(0.05, np.pi - 0.05, 200)])
targets = np.array([fk_points(q)[1] for q in qs])
for sign in [+1, -1]:
    back = np.array([fk_points(ik(p, sign))[1] for p in targets])
    check(f"200 個の目標で、sign={sign:+d} の解が目標に戻る", back, targets)

## 4. 2つの解を描く

**予想**:目標 $(1.2, 0.6)$ で、$q_2 > 0$ の解のとき、肘は「ベースから目標へ向かう線」の右側と左側のどちらに出るか。

- 予想:(ここに書く)

4つの象限に目標を1つずつ置き、2つの解を描く。点線は、ベースと目標を結ぶ線である。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for ax, target in zip(axes.ravel(), [(1.2, 0.6), (-1.0, 0.8), (-0.5, -1.0), (0.3, -1.2)]):
    for sign, color in [(+1, "C0"), (-1, "C1")]:
        elbow, hand = fk_points(ik(target, sign))
        draw_arm(ax, elbow, hand, color=color, label=f"sign={sign:+d}")
    ax.plot([0, target[0]], [0, target[1]], "k:", lw=1)
    ax.plot(*target, "k*", ms=12)
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect("equal"); ax.grid(True)
    ax.set_title(f"target {target}"); ax.legend(fontsize=8)
plt.show()

どの目標でも、$q_2 > 0$ の解(sign=+1)では、肘はベースから目標を見て**右側**(時計回りの側)に出る。$q_2 < 0$ の解では左側に出る。一方、「肘が上」になる解は、目標が右上にあるときと左下にあるときで入れ替わる。そのため、この単元では解を $q_2$ の符号で呼び分ける。

## 5. `atan2` と `arctan` の違い

**予想**:$q_1$ の式 (2) で、`np.arctan2(y, x)` の代わりに `np.arctan(y / x)` を使うと、目標 $(-1.0, 0.8)$ ではどうなるか(本文の確認課題3)。

- 予想:(ここに書く)

In [ ]:
def ik_arctan(p, sign=+1, l1=l1, l2=l2):
    # わざと間違えた版:目標の向きを arctan(y / x) で計算する
    x, y = p
    q2 = sign * np.arccos(cos_q2(p, l1, l2))
    q1 = np.arctan(y / x) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2))
    return np.array([q1, q2])

for target in [(1.0, 0.8), (-1.0, 0.8)]:
    print(f"目標 {target}:  arctan 版の手先 {np.round(fk_points(ik_arctan(target))[1], 4)}"
          f"   atan2 版の手先 {np.round(fk_points(ik(target))[1], 4)}")

$x > 0$ の目標では、どちらも正しい。$(-1.0, 0.8)$ では、`arctan` 版の手先は $(1.0, -0.8)$ になり、原点について反対側に行ってしまう。$\arctan(0.8 / (-1.0)) = \arctan(-0.8)$ は、左上ではなく右下の向きの角度を返すからである。

## 6. 境界での丸めの誤差

目標を、到達可能領域の外側の境界(半径 1.8)の上に置く。

**予想**:この目標で `ik` を呼ぶと、解はいくつ返ってくるか。

- 予想:(ここに書く)

In [ ]:
t = np.deg2rad(20)   # 目標の向き(20°)
p_edge = 1.8 * np.array([np.cos(t), np.sin(t)])
print("目標までの距離:", np.linalg.norm(p_edge))
print("c =", repr(cos_q2(p_edge)))
print("ik の結果:", ik(p_edge))

届くはずの目標なのに、丸めの誤差で $c$ が 1 をわずかに超え、`ik` は「届かない」と答えてしまう。

**実装**:$|c|$ が `1 + tol` 以下なら、`np.clip` で $c$ を $-1$ から $1$ の範囲にそろえてから解く関数 `ik_robust` を書く(本文 §5)。

In [ ]:
def ik_robust(p, sign=+1, l1=l1, l2=l2, tol=1e-9):
    x, y = p
    c = cos_q2(p, l1, l2)
    if abs(c) > 1 + tol:
        return None
    c = np.clip(c, -1.0, 1.0)
    q2 = sign * np.arccos(c)
    q1 = np.arctan2(y, x) - np.arctan2(l2 * np.sin(q2), l1 + l2 * np.cos(q2))
    return np.array([q1, q2])

check("境界の上の目標も解ける", fk_points(ik_robust(p_edge))[1], p_edge)
check("届かない目標は、今までどおり None", ik_robust([2.5, 0.0]) is None, True)

## 7. 目標を円に沿って動かす

**予想**:目標を、半径 1.2 の円に沿って一周させる。目標を動かすたびに $q_2 > 0$ の解を求めると、$q_1$ のグラフはどうなるか。

- 予想:(ここに書く)

In [ ]:
ts = np.linspace(0, 2 * np.pi, 200)
targets_c = 1.2 * np.column_stack([np.cos(ts), np.sin(ts)])
Q = np.array([ik_robust(p, +1) for p in targets_c])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(ts, Q[:, 0], label="q1"); axes[0].plot(ts, Q[:, 1], label="q2")
axes[0].set_title("ik as it is")
axes[1].plot(ts, np.unwrap(Q[:, 0]), label="q1 (np.unwrap)"); axes[1].plot(ts, Q[:, 1], label="q2")
axes[1].set_title("after np.unwrap")
for ax in axes:
    ax.set_xlabel("angle of the target [rad]"); ax.set_ylabel("joint angle [rad]"); ax.grid(True); ax.legend()
plt.show()
check("q1 が跳んでも、手先は目標どおり", [fk_points(q)[1] for q in Q], targets_c)

目標が x 軸の負の側を横切るところ(目標の向きが $\pi$ のところ)で、$q_1$ が約 $2\pi$ だけ跳ぶ。`atan2` が $-\pi$ から $\pi$ までの値しか返さないためで、手先は目標どおりに動いており、姿勢は跳んでいない。$q_2$ は、目標までの距離が一定なので変わらない。関節角を時系列として使うときは、`np.unwrap` でつなぎ直す。

## 8. 条件を変える

1. 目標を $(2.5, 0.0)$ にする。M3 §7 の1 では、数値的な方法で同じ目標を試した。解析解では何が分かり、数値的な方法との違いは何か
2. $l_1 = l_2 = 1.0$ にして、目標を原点 $(0, 0)$ にする。`ik_robust` は何を返すか。ほかの解はあるか
3. $q_2$ の可動範囲が $0 \le q_2 \le \pi$ のとき、2つの解のどちらが使えるか。目標 $(1.2, 0.6)$ で確かめる
4. 目標を $(1.8, 0.0)$(外側の境界)と $(0.2, 0.0)$(内側の境界)にすると、2つの解はどうなるか

In [ ]:
# 1. c > 1 なので解はない。解析解では「届かない」とすぐに分かる(数値的な方法は、収束しないことでしか分からない)
print("1. c =", cos_q2([2.5, 0.0]), "  ik:", ik_robust([2.5, 0.0]))

# 2. c = -1 なので q2 = π(折りたたむ)。このとき手先は q1 によらず原点にあるので、解は無数にある
q0 = ik_robust([0.0, 0.0], +1, l1=1.0, l2=1.0)
print("2. ik:", np.round(q0, 4), "  手先:", np.round(fk_points(q0, 1.0, 1.0)[1], 6))
for q1 in [0.0, 1.0, 2.0]:
    print(f"   q = ({q1}, π) の手先:", np.round(fk_points([q1, np.pi], 1.0, 1.0)[1], 6))

# 3. 0 <= q2 <= π なら、q2 > 0 の解(sign=+1)だけが使える
for sign in [+1, -1]:
    q = ik_robust([1.2, 0.6], sign)
    print(f"3. sign={sign:+d}: q = {np.round(q, 4)}  可動範囲の中か: {0 <= q[1] <= np.pi}")

# 4. 境界では c = ±1 なので q2 = ±0 または ±π となり、2つの解は同じ姿勢になる
for target in [(1.8, 0.0), (0.2, 0.0)]:
    print(f"4. 目標 {target}:", [np.round(ik_robust(target, s), 4) for s in [+1, -1]])

## 9. 説明(自分の言葉で書く)

1. 逆運動学の解が2つの場合、1つの場合、ない場合を、到達可能領域の図を使って説明する
   - (ここに書く)
2. $q_1$ を `np.arctan(y / x)` で計算してはいけないのはなぜか
   - (ここに書く)
3. 目標を円に沿って動かしたとき、$q_1$ のグラフが跳んだのはなぜか。アームの姿勢は跳んだか
   - (ここに書く)